# 01 — تجهيز البيانات والنموذج الأولي للبحث الدلالي
**Data preparation, first prototype, and early evaluation** — report Chapter Four, Steps 1-11, and Section 5.3.1.

هذا النوتبوك يبني من نص القرآن الملفين اللذين يعتمد عليهما الموقع: `verses.csv` (6236 آية) و`segments.csv`
(10675 جزءًا مقسّمًا بعلامات الوقف). ثم يجرّب أول نسخة من البحث الدلالي ويقيّمها.

- الخطوات 1-5: تحميل النص، التطبيع، التجزئة بعلامات الوقف، فصل البسملة، استبعاد الحروف المقطّعة.
- الخطوات 6-9: نموذج MiniLM الجاهز، التمثيلات، دالة البحث، واجهة أولية داخل النوتبوك.
- الخطوات 10-12: تقييم أولي (23 سؤالًا) ومقارنة مع Word2Vec وFastText، ومحاولة GloVe.
- الخطوة 13: حفظ الملفات.

**ملاحظة:** هذا هو النموذج الأولي. النظام النهائي يستبدل MiniLM بنموذج mpnet مدرَّب على بيانات قرآنية
(النوتبوك 02)، ويضيف مطابقة الكلمات بجذورها من مدوّنة القرآن الكريم (Quranic Arabic Corpus) والترتيب
الهجين — وهذا كله في `app.py` في المستودع.

**قبل التشغيل:** ارفع لجلسة Colab ملف `quran-uthmani.txt` من موقع [Tanzil](https://tanzil.net/download/)
(النص العثماني، صيغة Text مع علامات الوقف، Pause marks).

## 1) تحميل النص القرآني
الصيغة: `رقم_السورة|رقم_الآية|النص` لكل سطر. العدد المتوقع 6236 آية.

In [ ]:
verses = []
with open("quran-uthmani.txt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        sura, aya, text = line.split("|")
        verses.append((int(sura), int(aya), text))

print("عدد الآيات:", len(verses))
assert len(verses) == 6236

## 2) تطبيع النص العربي
- حذف الحركات، **ما عدا الألف الخنجرية** (U+0670) فتتحول إلى ألف عادية، لأنها حرف حقيقي في كلمات مثل
  `وَبِٱلْوَٰلِدَيْنِ`. أول نسخة كانت تحذفها، فانكسرت كلمة «الوالدين» وفشل سؤال «بر الوالدين» (التقرير، الخطوة 4).
- حذف علامات الضبط القرآنية الصغيرة، وتوحيد أشكال الألف، وحذف التطويل، وتوحيد المسافات.

In [ ]:
import re

arabic_diacritics = re.compile(r"[\u064B-\u065F\u0670\u06D6-\u06ED]")


def remove_diacritics(text):
    text = re.sub("\u0670", "\u0627", text)   # الألف الخنجرية -> ألف
    return arabic_diacritics.sub("", text)


def normalize_arabic(text):
    text = remove_diacritics(text)
    text = re.sub(r"[\u0610-\u061A\u06D6-\u06DC\u06DF-\u06E8\u06EA-\u06ED]", "", text)
    text = re.sub("[إأآٱ]", "ا", text)
    text = text.replace("ـ", "")
    return re.sub(r"\s+", " ", text).strip()


print(normalize_arabic("وَبِٱلْوَٰلِدَيْنِ"))   # يجب أن تطبع: وبالوالدين

# فحص: لا يبقى بعد التطبيع إلا حروف عربية ومسافات
chars = set()
for s, a, t in verses[:50]:
    chars.update(normalize_arabic(t))
print(sorted(chars))

## 3) التجزئة بعلامات الوقف (Tajweed Waqf segmentation)
علامات الوقف الست في اليونيكود U+06D6–U+06DB. الآية الطويلة تتقسم عندها إلى أجزاء، فيقدر السؤال يطابق
جزءًا مركّزًا من الآية بدل الآية كلها.

In [ ]:
waqf_pattern = re.compile(r"[\u06D6-\u06DB]")


def segment_by_waqf(text_raw):
    return [p.strip() for p in waqf_pattern.split(text_raw) if p.strip()]


counts = [len(segment_by_waqf(t)) for s, a, t in verses]
print("آيات بلا علامة وقف:", sum(c == 1 for c in counts))
print("أكثر عدد أجزاء في آية واحدة:", max(counts))
print(segment_by_waqf(verses[261][2]))

## 4) فصل البسملة
ملف Tanzil يلصق البسملة في بداية الآية الأولى من كل سورة (عدا الفاتحة والتوبة). اكتُشف هذا عند البحث عن
«النجم»: كان الجزء الأول من السورة بسملة وبداية الآية معًا. الحل: البسملة جزء مستقل.

In [ ]:
basmalah = [t for s, a, t in verses if s == 1 and a == 1][0]
print("سور تبدأ آيتها الأولى بالبسملة:", sum(1 for s, a, t in verses if a == 1 and s != 1 and t.startswith(basmalah)))


def segment_verse(s, a, text_raw):
    if a == 1 and s != 1 and text_raw.startswith(basmalah):
        return [basmalah] + segment_by_waqf(text_raw[len(basmalah):].strip())
    return segment_by_waqf(text_raw)


segments = [(s, a, i, part) for s, a, t in verses for i, part in enumerate(segment_verse(s, a, t), start=1)]
print("عدد الأجزاء:", len(segments))

## 5) استبعاد الحروف المقطّعة
الأجزاء التي هي حروف مقطّعة فقط («الم»، «طه»، «يس»...) قصيرة وغريبة لغويًا، فكانت تحصل على تشابه عالٍ مع
أسئلة لا علاقة لها بها. تُستبعد من الفهرس (30 جزءًا).

In [ ]:
muqattaat = {normalize_arabic(x) for x in
             ["الم", "المص", "الر", "المر", "كهيعص", "طه", "طسم", "طس", "يس", "ص", "حم", "عسق", "ق", "ن"]}
removed = [seg for seg in segments if normalize_arabic(seg[3]) in muqattaat]
segments = [seg for seg in segments if normalize_arabic(seg[3]) not in muqattaat]
print("المستبعد:", len(removed), "| الأجزاء النهائية:", len(segments))
assert len(segments) == 10675

## 6) النموذج الأولي: MiniLM متعدد اللغات
`paraphrase-multilingual-MiniLM-L12-v2` (384 بُعدًا). اختبار سلامة: جملتان متقاربتان معنًى يجب أن يكون
تشابههما أعلى من جملة بعيدة.

In [ ]:
!pip -q install sentence-transformers gensim

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
e1, e2, e3 = model.encode(["الحمد لله رب العالمين", "الشكر لله رب الخلائق", "السماء صافية اليوم"])
print("قريب بالمعنى:", round(util.cos_sim(e1, e2).item(), 3))
print("بعيد بالمعنى:", round(util.cos_sim(e1, e3).item(), 3))

## 7) التمثيلات (embeddings) للأجزاء وللآيات الكاملة
الآيات الكاملة تُستخدم فقط للمقارنة: هل البحث بالجزء أفضل من البحث بالآية كاملة؟

In [ ]:
segment_texts = [normalize_arabic(seg[3]) for seg in segments]
segment_embeddings = model.encode(segment_texts, batch_size=64, show_progress_bar=True)
verse_embeddings = model.encode([normalize_arabic(t) for s, a, t in verses], batch_size=64, show_progress_bar=True)
print(segment_embeddings.shape, verse_embeddings.shape)

## 8) دالة البحث الأولى
تشفير السؤال بنفس النموذج، ثم تشابه جيب التمام (cosine similarity) مع كل الأجزاء، وترتيب تنازلي.

In [ ]:
def search(query, top_k=5):
    q = model.encode(normalize_arabic(query))
    scores = util.cos_sim(q, segment_embeddings)[0]
    rows = []
    for idx in scores.argsort(descending=True)[:top_k]:
        s, a, part, text = segments[int(idx)]
        rows.append({"السورة": s, "الآية": a, "النص": text, "الدرجة": round(float(scores[idx]), 3)})
    return rows


for r in search("خلق السماوات والأرض"):
    print(r)

# فحص: جزء من القرآن كسؤال يجب أن يرجع نفسه أولًا بدرجة 1.0
print(search(segments[100][3], top_k=1))

## 9) الواجهة الأولية داخل النوتبوك (ipywidgets)
مربع نص، زر بحث، وجدول نتائج؛ ورسالة تنبيه إذا كان المربع فارغًا (التقرير، الشكلان 4.1 و4.2).

In [ ]:
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output

query_box = widgets.Text(placeholder="اكتب سؤالك هنا...", description="البحث:", layout=widgets.Layout(width="500px"))
search_button = widgets.Button(description="ابحث")
output = widgets.Output()


def on_search_click(_):
    with output:
        clear_output()
        if not query_box.value.strip():
            print("الرجاء إدخال نص للبحث")
            return
        display(pd.DataFrame(search(query_box.value)))


search_button.on_click(on_search_click)
display(query_box, search_button, output)

## 10) التقييم الأولي: 23 سؤالًا ومقياس MRR
لكل سؤال آية أو آيات صحيحة، ونسجّل رتبة أول آية صحيحة مرة بالآية كاملة ومرة بالجزء.
MRR = متوسط (1 ÷ الرتبة). هذه المجموعة يدوية وصغيرة، وهي مقياس أولي فقط (التقرير، الجدول 5.1).

In [ ]:
queries = [
    ("خلق السماوات والأرض", [(2, 117), (6, 101)]), ("الصبر على البلاء", [(2, 155), (2, 156), (2, 157)]),
    ("بر الوالدين", [(17, 23), (17, 24)]), ("الصدقة والإنفاق في سبيل الله", [(2, 261)]),
    ("التوبة والاستغفار", [(39, 53)]), ("المحافظة على الصلاة", [(2, 238)]),
    ("التيسير بعد العسر", [(94, 5), (94, 6)]), ("تنوع البشر وتعارفهم", [(49, 13)]),
    ("نعيم الجنة", [(47, 15)]), ("الإيمان بالغيب", [(2, 3)]), ("العدل بين الناس", [(16, 90)]),
    ("أداء الأمانة", [(4, 58)]), ("التوكل على الله في الرزق", [(65, 3)]), ("الموت حق على كل نفس", [(3, 185)]),
    ("الأخوة بين المؤمنين", [(49, 10)]), ("الصدق في القول", [(33, 70)]),
    ("الإحسان إلى الجار وذي القربى", [(4, 36)]), ("الحكمة والموعظة الحسنة في الدعوة", [(16, 125)]),
    ("شكر النعم والمزيد لمن شكر", [(14, 7)]), ("التفكر في خلق السماوات والأرض", [(3, 191)]),
    ("الوفاء بالعهد", [(17, 34)]), ("النهي عن الكبر والتباهي", [(31, 18)]), ("كفالة اليتيم والرفق به", [(93, 9)]),
]


def first_rank(scores, keys, relevant):
    for rank, idx in enumerate(scores.argsort(descending=True), start=1):
        if keys[int(idx)] in relevant:
            return rank
    return None


def mrr(ranks):
    return sum(1 / r for r in ranks if r) / len(ranks)


verse_keys = [(s, a) for s, a, t in verses]
segment_keys = [(s, a) for s, a, _, _ in segments]
full_ranks, seg_ranks = [], []
for q, rel in queries:
    qv = model.encode(normalize_arabic(q))
    full_ranks.append(first_rank(util.cos_sim(qv, verse_embeddings)[0], verse_keys, rel))
    seg_ranks.append(first_rank(util.cos_sim(qv, segment_embeddings)[0], segment_keys, rel))
    print(f"{q:<35} آية كاملة={full_ranks[-1]:<6} جزء={seg_ranks[-1]}")

print("\nMRR آية كاملة:", round(mrr(full_ranks), 3), "| MRR جزء:", round(mrr(seg_ranks), 3))
print("الجزء أفضل في", sum(s < f for s, f in zip(seg_ranks, full_ranks)), "سؤالًا")

## 11) مقارنة مع Word2Vec وFastText مدرّبين على نص القرآن فقط
تمثيل الآية = متوسط متجهات كلماتها. المتوقع أن يكونا أضعف من نموذج الجمل الجاهز، لأن نص القرآن صغير
(6236 آية) لتعلّم متجهات كلمات موثوقة، ولأن المتوسط يضيّع ترتيب الكلمات (التقرير، الجدول 5.2).

In [ ]:
import numpy as np
from gensim.models import FastText, Word2Vec

sentences = [normalize_arabic(t).split() for s, a, t in verses]


def verse_matrix(wv):
    rows = []
    for words in sentences:
        vecs = [wv[w] for w in words if w in wv]
        rows.append(np.mean(vecs, axis=0) if vecs else np.zeros(wv.vector_size))
    m = np.array(rows)
    return m / np.maximum(np.linalg.norm(m, axis=1, keepdims=True), 1e-9)


def rank_static(q, rel, wv, mat):
    vecs = [wv[w] for w in normalize_arabic(q).split() if w in wv]
    if not vecs:
        return None
    qv = np.mean(vecs, axis=0)
    order = np.argsort(-(mat @ (qv / np.linalg.norm(qv))))
    return next((r for r, i in enumerate(order, start=1) if verse_keys[i] in rel), None)


for name, cls in [("Word2Vec", Word2Vec), ("FastText", FastText)]:
    m = cls(sentences=sentences, vector_size=100, window=5, min_count=1, workers=4, sg=1)
    mat = verse_matrix(m.wv)
    ranks = [rank_static(q, rel, m.wv, mat) for q, rel in queries]
    print(f"MRR {name}:", round(mrr(ranks), 3), "| أقرب كلمات لـ«الله»:", [w for w, _ in m.wv.most_similar("الله", topn=5)])

## 12) محاولة GloVe (لم تكتمل)
مكتبة `glove-python-binary` لم تعد تُحدَّث وفشل تثبيتها في Colab. جُرّبت نسخة يدوية (مصفوفة تكرار مشترك
للكلمات ثم تقليص الأبعاد بـ SVD) وتوقفت عند خطأ في نوع البيانات، فوُثّقت المحاولة في التقرير كما هي.
ELMo وFlair لم يُدرَّبا، واستُبدلت المقارنة بنماذج جمل حديثة (النوتبوك 03).

## 13) حفظ الملفات
`segments.csv` و`verses.csv` هما نفس ملفي الموقع حرفيًا. تمثيلات MiniLM تُحفظ باسم مختلف لأن الموقع يستخدم
تمثيلات النموذج المدرَّب (النوتبوك 02).

In [ ]:
segments_df = pd.DataFrame(segments, columns=["sura", "aya", "part_num", "text"])
segments_df.to_csv("segments.csv", index=False, encoding="utf-8-sig")
pd.DataFrame(verses, columns=["sura", "aya", "text"]).to_csv("verses.csv", index=False, encoding="utf-8-sig")
np.save("segment_embeddings_minilm.npy", segment_embeddings.astype(np.float16))
print("الأجزاء:", len(segments_df))

from google.colab import files
files.download("segments.csv")
files.download("verses.csv")